# 02 — Silver SISMEPRE
## Transformación Bronze → Silver

Procesa las **7 tablas del modelo relacional SISMEPRE** (Sistema de Seguimiento de Metas del Impuesto Predial).

| Tabla | Descripción |
|-------|-------------|
| RENTAS_FORMULARIO | Instrumento de medición del IP |
| RENTAS_PREGUNTAS | Ítems y preguntas del formulario |
| RENTAS_RESPUESTAS | Respuestas reportadas por municipalidad |
| RENTAS_ESTADISTICA | Resumen estadístico por pregunta |
| RENTAS_ESAT_ESTADISTICA_ATM | Histórico de Administración Tributaria |
| RENTAS_ENTIDAD_ESTADO | Estado de registro de la entidad |
| RENTAS_ANO_APLICACION | Años de aplicación disponibles |

**Fuente:** `data/bronze/sismepre/{tabla}/batch_*.json`  
**Destino:** `data/silver/sismepre/{tabla}/` (Parquet particionado por `ANO_APLICACION` donde aplica)

**Transformaciones por tabla:**
- Limpieza de ghost nulls (todas)
- Casteos de tipos numéricos y fechas (tabla-específico)
- Particionamiento por `ANO_APLICACION` (tablas temporales)
- DQ Flags binarios (arquitectura pasiva — los datos malos NO se eliminan)


In [1]:
import sys, time
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, LongType, ByteType

from src.paths import BRONZE, SILVER, str_path, ensure_dirs
from src.spark_utils import get_spark, write_parquet
from src.transformations.common import clean_ghost_nulls, print_dq_summary
from core.audit.control_manager import ControlManager, ExecutionStatus

print("Imports OK")

Imports OK


In [2]:
# ── Tablas SISMEPRE a procesar ────────────────────────────────────────────────
SISMEPRE_TABLES = [
    "RENTAS_FORMULARIO",
    "RENTAS_PREGUNTAS",
    "RENTAS_RESPUESTAS",
    "RENTAS_ESTADISTICA",
    "RENTAS_ESAT_ESTADISTICA_ATM",
    "RENTAS_ENTIDAD_ESTADO",
    "RENTAS_ANO_APLICACION",
]

BRONZE_BASE = BRONZE["sismepre"]
SILVER_BASE = SILVER["sismepre"]

# Año de referencia para los DQ Flags de rango temporal
YEAR_MIN = 2011
YEAR_MAX = 2026

print(f"Bronze base: {BRONZE_BASE}")
print(f"Silver base: {SILVER_BASE}")
print(f"Tablas a procesar: {len(SISMEPRE_TABLES)}")

ensure_dirs()

Bronze base: /workspace/data/bronze/sismepre
Silver base: /workspace/data/silver/sismepre
Tablas a procesar: 7
[OK] Estructura de directorios verificada en /workspace/data


In [3]:
spark = get_spark(app_name="MEF_Silver_SISMEPRE", memory="4g")

[OK] SparkSession lista | version=3.5.0 | master=local[*]


## Lógica de Transformación por Tabla

Cada tabla tiene necesidades distintas porque modela una entidad diferente del dominio de impuesto predial.
El patrón de `part_col` define si la tabla se particiona por año al guardarse (lo cual acelera las consultas posteriores).

| Tabla | Tipos clave | Partición | DQ Flags |
|-------|-------------|-----------|----------|
| RENTAS_FORMULARIO | INT (años/periodo) | ANO_APLICACION | ESTADO_REGISTRO |
| RENTAS_PREGUNTAS | INT (años/periodo) | ANO_APLICACION | ESTADO_REGISTRO |
| RENTAS_RESPUESTAS | INT, DOUBLE, LONG, DATE | ANO_APLICACION | DEC_NEGATIVO, ESTADO |
| RENTAS_ESTADISTICA | INT (años) | ANO_APLICACION | ESTADO_REGISTRO |
| RENTAS_ESAT_ESTADISTICA_ATM | INT, DOUBLE (MON_*, NUM_*) | ANO_APLICACION | EMISION>BASE, AÑO_RANGO |
| RENTAS_ENTIDAD_ESTADO | INT (años) | ANO_APLICACION | ESTADO_INVALIDO |
| RENTAS_ANO_APLICACION | INT | — (catálogo pequeño) | — |


In [4]:
control = ControlManager(pipeline_name="silver_sismepre")
execution = control.start(input_parameters={"tables": SISMEPRE_TABLES})

metrics = {}      # {tabla: filas_escritas}
dq_registry = {}  # {tabla: [nombres_flags_aplicados]}
start_total = time.time()

try:
    for table_name in SISMEPRE_TABLES:
        print(f"\n{'─'*60}")
        print(f"Procesando: {table_name}")

        bronze_path = str_path(BRONZE_BASE / table_name.lower())
        silver_path = str_path(SILVER_BASE / table_name.lower())
        bronze_dir  = BRONZE_BASE / table_name.lower()

        # ── Verificar existencia de datos Bronze ──────────────────────────────
        if not bronze_dir.exists() or not any(bronze_dir.rglob("*.json")):
            print(f"   Sin datos Bronze en {bronze_dir} — saltando")
            metrics[table_name] = 0
            continue

        # ── Lectura (Spark infiere schema — los casteos se hacen abajo) ───────
        raw_df = spark.read.option("multiline", "false").json(bronze_path)
        n_raw = raw_df.count()
        print(f"   Registros Bronze: {n_raw:,}")

        # ── 1. Limpieza base: Ghost Nulls + trim/upper en _NOMBRE/_DESC ───────
        df = clean_ghost_nulls(raw_df)

        # Variables para lógica específica
        part_col  = None   # columna de partición (None = sin particionar)
        dq_cols   = []     # DQ Flags generados para esta tabla

        # ─────────────────────────────────────────────────────────────────────
        # ── 2. Lógica específica por tabla ────────────────────────────────────
        # ─────────────────────────────────────────────────────────────────────

        if table_name in ("RENTAS_ESAT_ESTADISTICA_ATM", "RENTAS_ESAT_ESTADISTICA"):
            # Tabla de Histórico de Administración Tributaria Municipal
            # Contiene columnas monetarias (MON_*) y de conteo (NUM_*) que llegan
            # como string desde la API — se castean a DOUBLE para operaciones math.
            part_col = "ANO_APLICACION"

            # Casteos numéricos por prefijo de columna
            mon_num_cols = [c for c in df.columns if c.startswith("MON_") or c.startswith("NUM_")]
            for c in mon_num_cols:
                df = df.withColumn(c, F.col(c).cast(DoubleType()))

            # Casteos de dimensiones temporales
            for c in ["ANO_APLICACION", "ANO_ESTADISTICA", "MES_ESTADISTICA", "PERIODO"]:
                if c in df.columns:
                    df = df.withColumn(c, F.col(c).cast(IntegerType()))

            # DQ Flag 1: La emisión no puede superar la base imponible
            if "MON_EMISIONPREDIAL_AFECTO" in df.columns and "MON_BASEIMPONIBLE_AFECTO" in df.columns:
                df = df.withColumn(
                    "DQ_EMISION_SUPERA_BASE",
                    F.when(
                        F.col("MON_EMISIONPREDIAL_AFECTO").isNotNull() &
                        F.col("MON_BASEIMPONIBLE_AFECTO").isNotNull() &
                        (F.col("MON_EMISIONPREDIAL_AFECTO") > F.col("MON_BASEIMPONIBLE_AFECTO")),
                        1
                    ).otherwise(0).cast(ByteType())
                )
                dq_cols.append("DQ_EMISION_SUPERA_BASE")

            # DQ Flag 2: El año de aplicación debe estar dentro del rango esperado
            if "ANO_APLICACION" in df.columns:
                df = df.withColumn(
                    "DQ_ANO_FUERA_RANGO",
                    F.when(
                        F.col("ANO_APLICACION").isNull() |
                        (F.col("ANO_APLICACION") < YEAR_MIN) |
                        (F.col("ANO_APLICACION") > YEAR_MAX),
                        1
                    ).otherwise(0).cast(ByteType())
                )
                dq_cols.append("DQ_ANO_FUERA_RANGO")

        elif table_name == "RENTAS_RESPUESTAS":
            # Tabla de respuestas por municipalidad — polimórfica:
            # una misma pregunta puede tener respuesta decimal, entera o de fecha
            part_col = "ANO_APLICACION"

            for c in ["ANO_APLICACION", "PERIODO"]:
                if c in df.columns:
                    df = df.withColumn(c, F.col(c).cast(IntegerType()))

            # Cada tipo de respuesta tiene su propio tipo de dato
            if "RESPUESTA_DECIMAL" in df.columns:
                df = df.withColumn("RESPUESTA_DECIMAL", F.col("RESPUESTA_DECIMAL").cast(DoubleType()))
            if "RESPUESTA_ENTERO" in df.columns:
                df = df.withColumn("RESPUESTA_ENTERO", F.col("RESPUESTA_ENTERO").cast(LongType()))
            if "RESPUESTA_FECHA" in df.columns:
                # La API entrega fechas como string ISO ('YYYY-MM-DD')
                df = df.withColumn("RESPUESTA_FECHA", F.to_date(F.col("RESPUESTA_FECHA")))

            # DQ Flag 1: Los valores decimales no deberían ser negativos en este dominio
            if "RESPUESTA_DECIMAL" in df.columns:
                df = df.withColumn(
                    "DQ_DECIMAL_NEGATIVO",
                    F.when(F.col("RESPUESTA_DECIMAL") < 0, 1).otherwise(0).cast(ByteType())
                )
                dq_cols.append("DQ_DECIMAL_NEGATIVO")

            # DQ Flag 2: Solo se aceptan estados 'A' (Activo) o 'I' (Inactivo)
            if "ESTADO_REGISTRO" in df.columns:
                df = df.withColumn(
                    "DQ_ESTADO_REGISTRO_INVALIDO",
                    F.when(
                        F.col("ESTADO_REGISTRO").isNotNull() &
                        ~F.col("ESTADO_REGISTRO").isin("A", "I"),
                        1
                    ).otherwise(0).cast(ByteType())
                )
                dq_cols.append("DQ_ESTADO_REGISTRO_INVALIDO")

        elif table_name == "RENTAS_ENTIDAD_ESTADO":
            # Estado de registro de cada municipalidad en el sistema SISMEPRE
            part_col = "ANO_APLICACION"

            for c in ["ANO_APLICACION", "PERIODO"]:
                if c in df.columns:
                    df = df.withColumn(c, F.col(c).cast(IntegerType()))

            # DQ Flag: Estados válidos del sistema son solo 'I' (Inicio) o 'A' (Activo)
            if "ESTADO" in df.columns:
                df = df.withColumn(
                    "DQ_ESTADO_INVALIDO",
                    F.when(
                        F.col("ESTADO").isNotNull() &
                        ~F.col("ESTADO").isin("I", "A"),
                        1
                    ).otherwise(0).cast(ByteType())
                )
                dq_cols.append("DQ_ESTADO_INVALIDO")

        elif table_name in ("RENTAS_FORMULARIO", "RENTAS_PREGUNTAS", "RENTAS_ESTADISTICA"):
            # Tablas de estructura/catálogo del formulario — principalmente descriptivas
            # pero con dimensión temporal (por año de aplicación)
            part_col = "ANO_APLICACION"

            for c in ["ANO_APLICACION", "PERIODO", "ANO_ESTADISTICA"]:
                if c in df.columns:
                    df = df.withColumn(c, F.col(c).cast(IntegerType()))

            # DQ Flag: Estado del registro debe ser 'A' o 'I'
            if "ESTADO_REGISTRO" in df.columns:
                df = df.withColumn(
                    "DQ_ESTADO_REGISTRO_INVALIDO",
                    F.when(
                        F.col("ESTADO_REGISTRO").isNotNull() &
                        ~F.col("ESTADO_REGISTRO").isin("A", "I"),
                        1
                    ).otherwise(0).cast(ByteType())
                )
                dq_cols.append("DQ_ESTADO_REGISTRO_INVALIDO")

        elif table_name == "RENTAS_ANO_APLICACION":
            # Tabla de catálogo pequeña — solo lista los años disponibles.
            # No se particiona (tamaño minúsculo) pero sí se castea el año.
            if "ANO_APLICACION" in df.columns:
                df = df.withColumn("ANO_APLICACION", F.col("ANO_APLICACION").cast(IntegerType()))
            # Sin DQ flags (es un catálogo de referencia controlado por el sistema)

        # ── 3. Mostrar resumen de DQ si hay flags ─────────────────────────────
        if dq_cols:
            print_dq_summary(df, dq_cols)
            dq_registry[table_name] = dq_cols

        # ── 4. Escritura Parquet (con partición si aplica) ────────────────────
        # Nota: write_parquet() ya gestiona el modo overwrite y el partitionBy
        partition = [part_col] if part_col and part_col in df.columns else None
        n_written = write_parquet(df, silver_path, mode="overwrite", partition_by=partition)
        metrics[table_name] = n_written

    elapsed = time.time() - start_total
    control.end(
        status=ExecutionStatus.SUCCESS,
        output_summary={**metrics, "elapsed_sec": round(elapsed, 1)}
    )
    print(f"\nSilver SISMEPRE completado en {elapsed:.1f}s")

except Exception as e:
    control.log_error("SilverSismepreError", str(e))
    control.end(status=ExecutionStatus.FAILED, output_summary={"error": str(e)})
    raise

2026-06-19 11:03:32 | INFO     | mef_dw.audit.control_manager | [AUDIT] Ejecución iniciada | pipeline=silver_sismepre id=f0107b77

────────────────────────────────────────────────────────────
Procesando: RENTAS_FORMULARIO
   Registros Bronze: 98

📊 Resumen de Quality Flags:
  Flag                             Registros con flag
  ----------------------------------------------------
  DQ_ESTADO_REGISTRO_INVALIDO             0  (0.0%)
  ✅ 98 filas → /workspace/data/silver/sismepre/rentas_formulario

────────────────────────────────────────────────────────────
Procesando: RENTAS_PREGUNTAS
   Registros Bronze: 836

📊 Resumen de Quality Flags:
  Flag                             Registros con flag
  ----------------------------------------------------
  DQ_ESTADO_REGISTRO_INVALIDO             0  (0.0%)
  ✅ 836 filas → /workspace/data/silver/sismepre/rentas_preguntas

────────────────────────────────────────────────────────────
Procesando: RENTAS_RESPUESTAS
   Registros Bronze: 174,210

📊 Resu

In [5]:
# ── Resumen final ─────────────────────────────────────────────────────────────
print("\nResumen Silver SISMEPRE:")
print(f"  {'Tabla':<35} {'Filas':>10}  {'DQ Flags'}")
print("  " + "─" * 70)
for table, count in metrics.items():
    flags  = ", ".join(dq_registry.get(table, [])) or "—"
    print(f"  {table:<33} {count:>10,}  {flags}")
print(f"\n  Total filas Silver: {sum(metrics.values()):,}")


Resumen Silver SISMEPRE:
  Tabla                                    Filas  DQ Flags
  ──────────────────────────────────────────────────────────────────────
  RENTAS_FORMULARIO                         98  DQ_ESTADO_REGISTRO_INVALIDO
  RENTAS_PREGUNTAS                         836  DQ_ESTADO_REGISTRO_INVALIDO
  RENTAS_RESPUESTAS                    174,210  DQ_DECIMAL_NEGATIVO, DQ_ESTADO_REGISTRO_INVALIDO
  RENTAS_ESTADISTICA                       233  DQ_ESTADO_REGISTRO_INVALIDO
  RENTAS_ESAT_ESTADISTICA_ATM          134,144  DQ_EMISION_SUPERA_BASE, DQ_ANO_FUERA_RANGO
  RENTAS_ENTIDAD_ESTADO                 19,037  DQ_ESTADO_INVALIDO
  RENTAS_ANO_APLICACION                     26  —

  Total filas Silver: 328,584


In [6]:
# ── Verificación: leer de vuelta y mostrar schema + conteos ──────────────────
print("\n🔍 Verificación de Parquet escritos:")
for table_name in SISMEPRE_TABLES:
    silver_path = str_path(SILVER_BASE / table_name.lower())
    try:
        df_check = spark.read.parquet(silver_path)
        n = df_check.count()
        cols = len(df_check.columns)
        has_partition = "ANO_APLICACION" in df_check.columns
        part_str = "(particionado por ANO_APLICACION)" if has_partition else "(sin partición)"
        print(f"  {table_name:<35} {n:>10,} filas | {cols} cols {part_str}")
    except Exception as ex:
        print(f"  {table_name:<35} ERROR: {ex}")


🔍 Verificación de Parquet escritos:
  RENTAS_FORMULARIO                           98 filas | 11 cols (particionado por ANO_APLICACION)
  RENTAS_PREGUNTAS                           836 filas | 16 cols (particionado por ANO_APLICACION)
  RENTAS_RESPUESTAS                      174,210 filas | 13 cols (particionado por ANO_APLICACION)
  RENTAS_ESTADISTICA                         233 filas | 8 cols (particionado por ANO_APLICACION)
  RENTAS_ESAT_ESTADISTICA_ATM            134,144 filas | 44 cols (particionado por ANO_APLICACION)
  RENTAS_ENTIDAD_ESTADO                   19,037 filas | 15 cols (particionado por ANO_APLICACION)
  RENTAS_ANO_APLICACION                       26 filas | 9 cols (particionado por ANO_APLICACION)
